# Fine-tune E5 cho Vietnamese Legal RAG
Chạy trên T4 GPU — ~30 phút

In [ ]:
# 1. Mount Google Drive (lưu model vào đây để không bị mất)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Cài dependencies
!pip install -q sentence-transformers datasets accelerate

In [ ]:
# 3. Upload dataset
from google.colab import files
uploaded = files.upload()  # upload legal_triplets_dataset.zip

In [ ]:
# 4. Giải nén dataset
!unzip -q legal_triplets_dataset.zip
!ls legal_triplets_dataset/

In [ ]:
# 5. Train
import inspect, os
from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from datasets import load_from_disk

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

dataset = load_from_disk('legal_triplets_dataset')
print(f'Triplets: {len(dataset)}')

model = SentenceTransformer('intfloat/multilingual-e5-base', device=device)

examples = [
    InputExample(texts=[s['anchor'], s['positive'], s['negative']])
    for s in dataset
]
train_dataloader = DataLoader(examples, shuffle=True, batch_size=32)

sig = inspect.signature(losses.TripletLoss.__init__)
if 'triplet_margin' in sig.parameters:
    loss = losses.TripletLoss(model=model, triplet_margin=0.5)
else:
    loss = losses.TripletLoss(model=model, margin=0.5)

EPOCHS = 15
total_steps = len(train_dataloader) * EPOCHS
warmup_steps = int(total_steps * 0.1)
print(f'Total steps: {total_steps}, warmup: {warmup_steps}')

OUTPUT = 'drive/MyDrive/finetuned-embedder'  # lưu vào Google Drive
Path(OUTPUT).mkdir(parents=True, exist_ok=True)

if hasattr(model, 'old_fit'):
    model.old_fit(
        train_objectives=[(train_dataloader, loss)],
        epochs=EPOCHS,
        warmup_steps=warmup_steps,
        output_path=OUTPUT,
        save_best_model=False,
        show_progress_bar=True,
        optimizer_params={'lr': 2e-5},
        weight_decay=1e-6,
        checkpoint_path=None,
        checkpoint_save_steps=0,
        checkpoint_save_total_limit=0,
    )
else:
    model.fit(
        train_objectives=[(train_dataloader, loss)],
        epochs=EPOCHS,
        warmup_steps=warmup_steps,
        output_path=OUTPUT,
        save_best_model=False,
        show_progress_bar=True,
        optimizer_params={'lr': 2e-5},
    )

print('\n✓ Training xong! Model đã lưu vào Google Drive.')

In [ ]:
# 6. Zip và download về máy
!zip -r finetuned-embedder.zip drive/MyDrive/finetuned-embedder/
from google.colab import files
files.download('finetuned-embedder.zip')
print('✓ Download xong — giải nén vào models/finetuned-embedder/ trên máy')